In [1]:
!pip -q install playwright
!playwright install chromium
!apt-get update
!apt-get install -y libatk1.0-0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 19.0 MB/s eta 0:00:00
186.8 MiB [] 0% 0.0s186.8 MiB [] 0% 30.5s186.8 MiB [] 0% 14.5s186.8 MiB [] 0% 7.6s186.8 MiB [] 1% 4.3s186.8 MiB [] 2% 3.5s186.8 MiB [] 3% 3.1s186.8 MiB [] 4% 2.8s186.8 MiB [] 5% 2.6s186.8 MiB [] 6% 2.4s186.8 MiB [] 6% 2.6s186.8 MiB [] 7% 2.7s186.8 MiB [] 7% 2.9s186.8 MiB [] 7% 3.0s186.8 MiB [] 8% 2.8s186.8 MiB [] 9% 2.7s186.8 MiB [] 10% 2.5s186.8 MiB [] 11% 2.4s186.8 MiB [] 12% 2.3s186.8 MiB [] 13% 2.2s186.8 MiB [] 14% 2.2s186.8 MiB [] 14% 2.1s186.8 MiB [] 15% 2.1s186.8 MiB [] 16% 2.0s186.8 MiB [] 17% 2.0s186.8 MiB [] 18% 2.0s186.8 MiB [] 19% 1.9s186.8 MiB [] 20% 1.9s186.8 MiB [] 20% 2.0s186.8 MiB [] 21% 2.0s186.8 MiB [] 22% 2.0s186.8 MiB [] 23% 1.9s186.8 MiB [] 24% 1.9s186.8 MiB [] 25% 1.9s186.8 MiB [] 26% 1.9s186.8 MiB [] 28% 1.8s186.8 MiB [] 29% 1.7s186.8 MiB [] 30% 1.7s186.8 MiB [] 32% 1.6s186.8 MiB [] 33% 1.5s186.8 MiB [] 34% 1.5s186.8 MiB [] 35% 1.4s186.8 MiB [] 37% 1.4s186.8 MiB [] 38% 1.3s186.8 MiB [

In [2]:
!apt-get update -qq

!apt-get install -y \
    libatk-bridge2.0-0 \
    libatk1.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libnss3 \
    libasound2t64 \
    libpango-1.0-0 \
    libcairo2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'libatk-bridge2.0-0t64' instead of 'libatk-bridge2.0-0'
Note, selecting 'libatk1.0-0t64' instead of 'libatk1.0-0'
Note, selecting 'libcups2t64' instead of 'libcups2'
libatk1.0-0t64 is already the newest version (2.52.0-1build1).
libcups2t64 is already the newest version (2.4.7-1.2ubuntu7.14).
libcups2t64 set to manually installed.
libxkbcommon0 is already the newest version (1.6.0-1build1).
libxkbcommon0 set to manually installed.
libxdamage1 is already the newest version (1:1.1.6-1build1).
libxdamage1 set to manually installed.
libxfixes3 is already the newest version (1:6.0.0-2build1).
libxfixes3 set to manually installed.
libxrandr2 is already the newest version (2:1.5.2-2build1).
libxr

In [3]:
import asyncio
import base64
import json
import re
from pathlib import Path

from playwright.async_api import async_playwright


MEDICAMENTOS = [
    "METOTREXATO",
    "ADALIMUMABE",
    "ETANERCEPTE",
    "SALBUTAMOL",
    "BUDESONIDA",
    "INSULINA HUMANA REGULAR",
    "INSULINA HUMANA NPH",
    "METFORMINA",
    "SINVASTATINA",
    "TENOFOVIR",
    "ENTECAVIR",
    "SOFOSBUVIR",
    "LOSARTANA",
    "ENALAPRIL",
    "HIDROCLOROTIAZIDA",
    "DOLUTEGRAVIR",
]

ALIASES = {
    "LOSARTANA": [
        "LOSARTANA POTÁSSICA",
        "LOSARTANA POTASSICA",
    ],

    "DOLUTEGRAVIR": [
        "DOLUTEGRAVIR SÓDICO",
        "DOLUTEGRAVIR SODICO",
    ],

    "METFORMINA": [
        "CLORIDRATO DE METFORMINA",
        "GLIFAGE",
    ],

    "ENALAPRIL": [
        "MALEATO DE ENALAPRIL",
    ],

    "SALBUTAMOL": [
        "SULFATO DE SALBUTAMOL",
    ],

    "TENOFOVIR": [
        "FUMARATO DE TENOFOVIR DESOPROXILA",
        "TENOFOVIR DESOPROXILA",
    ],
}

PASTA_SAIDA = Path("/content/bulas")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

BASE = "https://consultas.anvisa.gov.br"

API_BULARIO = (
    BASE
    + "/api/consulta/bulario"
)

API_PDF = (
    BASE
    + "/api/consulta/medicamentos/arquivo/bula/parecer/"
)

In [4]:
def nome_aceito(nome_produto, medicamento):
    nome_produto = nome_produto.strip().upper()
    medicamento = medicamento.strip().upper()

    if nome_produto == medicamento:
        return True

    aliases = ALIASES.get(medicamento, [])

    return nome_produto in aliases

In [5]:
def limpar_nome(texto):
    texto = texto.strip().lower() # Converte para minúsculas para padronizar nomes de pastas
    texto = re.sub(r'[\\/:*?"<>|]', "_", texto)
    texto = re.sub(r"\s+", "_", texto)
    return texto

In [6]:
async def baixar_pdf(page, token, destino):
    url = f"{API_PDF}{token}/?Authorization="

    resultado = await page.evaluate(
        """
        async (url) => {

            const response = await fetch(url, {
                method: "GET",
                credentials: "include"
            });

            if (!response.ok) {
                return {
                    ok: false,
                    status: response.status,
                    erro: await response.text()
                };
            }

            const buffer = await response.arrayBuffer();

            const bytes = new Uint8Array(buffer);

            let binary = "";

            const chunk = 8192;

            for (let i = 0; i < bytes.length; i += chunk) {

                binary += String.fromCharCode(
                    ...bytes.subarray(i, i + chunk)
                );
            }

            return {
                ok: true,
                status: response.status,
                base64: btoa(binary),
                contentType:
                    response.headers.get("content-type")
            };
        }
        """,
        url
    )

    if not resultado["ok"]:
        print(
            f"ERRO PDF HTTP {resultado['status']}"
        )

        print(
            resultado.get("erro", "")[:500]
        )

        return False

    dados = base64.b64decode(
        resultado["base64"]
    )

    destino.write_bytes(dados)

    print(
        f"   OK: {destino.name} "
        f"({len(dados)/1024:.1f} KB)"
    )

    return True

In [7]:
async def buscar_medicamento(page, medicamento):

    termos = [medicamento] + ALIASES.get(
        medicamento.upper(),
        []
    )

    for termo in termos:

        print(f"Tentando busca: {termo}")

        dados = await buscar_json_no_navegador(
            page,
            termo
        )

        if not dados:
            continue

        resultados = dados.get("content", [])

        if resultados:
            print(
                f"Encontrado usando: {termo}"
            )

            return resultados

    return []

In [8]:
async def processar_medicamento(page, medicamento):

    print("\n" + "=" * 60)
    print(f"Buscando: {medicamento}")
    print("=" * 60)

    resultados = await buscar_medicamento(
    page,
    medicamento
    )
    print(
        f"Resultados retornados pela ANVISA: "
        f"{len(resultados)}"
    )

    # Correspondência exata
    resultados_exatos = [
    item
    for item in resultados
    if nome_aceito(
        item.get("nomeProduto", ""),
        medicamento
    )
]
    if resultados_exatos:
        resultados_exatos = [resultados_exatos[0]]

    if not resultados_exatos:

        print(
            "Nenhuma correspondência exata."
        )

        print("Encontrados:")

        for item in resultados:
            print(
                " -",
                item.get("nomeProduto")
            )

        return

    for item in resultados_exatos:

        nome = item["nomeProduto"]
        registro = item["numeroRegistro"]
        empresa = item.get(
            "razaoSocial",
            "empresa_desconhecida"
        )

        print()
        print(f"Produto : {nome}")
        print(f"Registro: {registro}")
        print(f"Empresa : {empresa}")

        pasta = (
            PASTA_SAIDA /
            limpar_nome(nome)
        )

        pasta.mkdir(
            parents=True,
            exist_ok=True
        )

        token_paciente = item.get(
            "idBulaPacienteProtegido"
        )

        token_profissional = item.get(
            "idBulaProfissionalProtegido"
        )

        if token_paciente:

            destino = (
                pasta /
                f"{registro}_paciente.pdf"
            )

            await baixar_pdf(
                page,
                token_paciente,
                destino
            )

        if token_profissional:

            destino = (
                pasta /
                f"{registro}_profissional.pdf"
            )

            await baixar_pdf(
                page,
                token_profissional,
                destino
            )

        # Salva também metadados
        metadata = {
            "idProduto":
                item.get("idProduto"),

            "numeroRegistro":
                registro,

            "nomeProduto":
                nome,

            "expediente":
                item.get("expediente"),

            "razaoSocial":
                empresa,

            "cnpj":
                item.get("cnpj"),

            "numeroTransacao":
                item.get("numeroTransacao"),

            "data":
                item.get("data"),

            "numProcesso":
                item.get("numProcesso"),

            "dataAtualizacao":
                item.get("dataAtualizacao"),
        }

        with open(
            pasta / "metadata.json",
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                metadata,
                f,
                ensure_ascii=False,
                indent=2
            )

In [9]:
from urllib.parse import quote


async def buscar_json_no_navegador_OLD(page, medicamento):

    url = (
        "https://consultas.anvisa.gov.br/"
        "#/bulario/q/?nomeProduto="
        + quote(medicamento)
    )

    print(f"Abrindo pesquisa: {medicamento}")

    try:
        async with page.expect_response(
            lambda response:
                "/api/consulta/bulario?" in response.url,
            timeout=60000
        ) as response_info:

            await page.goto(
                url,
                wait_until="domcontentloaded",
                timeout=60000
            )

        response = await response_info.value

        print("API:", response.status)
        print("URL:", response.url)

        if response.status != 200:
            texto = await response.text()
            print(texto[:1000])
            return None

        dados = await response.json()

        return dados

    except Exception as e:
        print("Erro capturando resposta:", e)
        return None

In [10]:
import asyncio
import time
from urllib.parse import quote


async def buscar_json_no_navegador(
    page,
    medicamento,
    tentativas=3
):

    for tentativa in range(1, tentativas + 1):

        # parâmetro variável para evitar reaproveitamento/cache
        cache_buster = int(time.time() * 1000)

        url = (
            "https://consultas.anvisa.gov.br/"
            "#/bulario/q/"
            f"?nomeProduto={quote(medicamento)}"
            f"&_={cache_buster}"
        )

        print(f"Abrindo pesquisa: {medicamento}")
        print(
            f"Tentativa {tentativa}/{tentativas}"
        )

        try:

            async with page.expect_response(
                lambda response:
                    "/api/consulta/bulario?" in response.url,
                timeout=30000
            ) as response_info:

                # primeiro sai da pesquisa atual
                await page.goto(
                    "https://consultas.anvisa.gov.br/#/bulario/",
                    wait_until="domcontentloaded",
                    timeout=60000
                )

                await page.wait_for_timeout(1000)

                # depois abre a nova pesquisa
                await page.goto(
                    url,
                    wait_until="domcontentloaded",
                    timeout=60000
                )

            response = await response_info.value

            print("API:", response.status)
            print("URL:", response.url)

            if response.status == 200:

                dados = await response.json()

                return dados

            texto = await response.text()

            print(
                f"Erro da API: HTTP {response.status}"
            )

            print(texto[:500])

        except Exception as e:

            print(
                f"Erro capturando resposta: {e}"
            )

        if tentativa < tentativas:

            espera = tentativa * 5

            print(
                f"Aguardando {espera}s "
                "antes de tentar novamente..."
            )

            await asyncio.sleep(espera)

    return None

In [11]:
async def main():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
            ]
        )

        context = await browser.new_context(
            locale="pt-BR",
            user_agent=(
                "Mozilla/5.0 "
                "(Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/153.0.0.0 "
                "Safari/537.36"
            )
        )

        page = await context.new_page()

        print(
            "Abrindo o Bulário da ANVISA..."
        )

        response = await page.goto(
            "https://consultas.anvisa.gov.br/#/bulario/",
            wait_until="domcontentloaded",
            timeout=60000
        )

        print(
            "Página aberta:",
            response.status if response else "?"
        )

        await page.wait_for_timeout(5000)

        for medicamento in MEDICAMENTOS:

            try:

                await processar_medicamento(
                    page,
                    medicamento
                )

            except Exception as e:

                print(
                    f"ERRO em {medicamento}: {e}"
                )

            # Espera 5 segundos antes do proximo medicamento
            print("\nAguardando 5 segundos...")
            await asyncio.sleep(5)
        await browser.close()
        print("Finalizado!")


await main()

Abrindo o Bulário da ANVISA...
Página aberta: 200

Buscando: METOTREXATO
Tentando busca: METOTREXATO
Abrindo pesquisa: METOTREXATO
Tentativa 1/3
API: 200
URL: https://consultas.anvisa.gov.br/api/consulta/bulario?column=&count=10&filter%5BnomeProduto%5D=METOTREXATO&order=asc&page=1
Encontrado usando: METOTREXATO
Resultados retornados pela ANVISA: 1

Produto : metotrexato
Registro: 116370142
Empresa : BLAU FARMACÊUTICA S.A.
   OK: 116370142_paciente.pdf (1147.0 KB)
   OK: 116370142_profissional.pdf (1026.8 KB)

Aguardando 5 segundos...

Buscando: ADALIMUMABE
Tentando busca: ADALIMUMABE
Abrindo pesquisa: ADALIMUMABE
Tentativa 1/3
Erro capturando resposta: Timeout 30000ms exceeded while waiting for event "response"
Aguardando 5s antes de tentar novamente...
Abrindo pesquisa: ADALIMUMABE
Tentativa 2/3
Erro capturando resposta: Timeout 30000ms exceeded while waiting for event "response"
Aguardando 10s antes de tentar novamente...
Abrindo pesquisa: ADALIMUMABE
Tentativa 3/3
API: 200
URL: http

In [14]:
# Comprime a pasta 'bulas' em um arquivo ZIP
!zip -r /content/bulas_comprimidas.zip /content/bulas

print("Todas as bulas foram compactadas em: /content/bulas_comprimidas.zip")

  adding: content/bulas/ (stored 0%)
  adding: content/bulas/hidroclorotiazida/ (stored 0%)
  adding: content/bulas/hidroclorotiazida/110390200_profissional.pdf (deflated 12%)
  adding: content/bulas/hidroclorotiazida/metadata.json (deflated 32%)
  adding: content/bulas/sulfato_de_salbutamol/ (stored 0%)
  adding: content/bulas/sulfato_de_salbutamol/metadata.json (deflated 33%)
  adding: content/bulas/sulfato_de_salbutamol/103700792_paciente.pdf (deflated 13%)
  adding: content/bulas/metotrexato/ (stored 0%)
  adding: content/bulas/metotrexato/116370142_profissional.pdf (deflated 4%)
  adding: content/bulas/metotrexato/116370142_paciente.pdf (deflated 4%)
  adding: content/bulas/metotrexato/metadata.json (deflated 34%)
  adding: content/bulas/sinvastatina/ (stored 0%)
  adding: content/bulas/sinvastatina/118190226_profissional.pdf (deflated 13%)
  adding: content/bulas/sinvastatina/metadata.json (deflated 33%)
  adding: content/bulas/maleato_de_enalapril/ (stored 0%)
  adding: content/

In [15]:
import os
import shutil

output_zip_path = "/content/bulas_comprimidas.zip"
base_dir = "/content/bulas"

# Get all subdirectories in base_dir
all_subdirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

# Filter for subdirectories with all lowercase names
lowercase_subdirs = [d for d in all_subdirs if d.islower()]

# Create a temporary directory to copy only the lowercase folders into
temp_dir_for_zip = "/content/bulas_temp_for_zip"
os.makedirs(temp_dir_for_zip, exist_ok=True)

print(f"Pastas em minúsculas a serem compactadas: {lowercase_subdirs}")

# Copy only the lowercase folders to the temporary directory
for subdir in lowercase_subdirs:
    shutil.copytree(os.path.join(base_dir, subdir), os.path.join(temp_dir_for_zip, subdir))

# Create the zip archive from the temporary directory
shutil.make_archive(output_zip_path.replace('.zip', ''), 'zip', temp_dir_for_zip)

# Remove the temporary directory
shutil.rmtree(temp_dir_for_zip)

print(f"Todas as pastas com nomes em minúsculas foram compactadas em: {output_zip_path}")

Pastas em minúsculas a serem compactadas: ['hidroclorotiazida', 'sulfato_de_salbutamol', 'metotrexato', 'sinvastatina', 'maleato_de_enalapril', 'entecavir', 'budesonida', 'dolutegravir_sódico', 'fumarato_de_tenofovir_desoproxila', 'sofosbuvir', 'cloridrato_de_metformina', 'losartana_potássica']
Todas as pastas com nomes em minúsculas foram compactadas em: /content/bulas_comprimidas.zip
